# Data Ingestion & Quality Checks — `customers_raw`

Demonstrates the full journey from raw CSV to a validated Postgres table:

1. Load the raw CSV and inspect the data
2. Run the five data-quality gates against the in-memory DataFrame
3. Ingest to Postgres via `ingest()` — which enforces the gates internally
4. Confirm the data landed correctly in `customers_raw`

| Gate | Check | Severity |
|---|---|---|
| 1 | Schema — column presence, types, value ranges, categoricals | ERROR |
| 2 | Duplicate `customerid` values | ERROR |
| 3 | Binary churn labels, no missing values | ERROR |
| 4 | Unexpected NULL `totalcharges` (non-zero tenure) | WARNING |
| 5 | Row count ≥ 1 000; critical-column null rates ≤ 5 % | ERROR / WARNING |

**Requires:** `docker compose --profile infra up -d` and `POSTGRES_URL` in `.env`.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from omegaconf import OmegaConf
from sqlalchemy import text

from telco_churn.data.ingest import ingest, load_raw_csv
from telco_churn.data.validate import validate_raw
from telco_churn.utils.db import get_engine
from telco_churn.utils.logging import configure_logging

# nbconvert sets cwd to notebooks/; this normalizes to project root
if not Path("configs").exists():
    os.chdir("..")

load_dotenv()
configure_logging()

cfg = OmegaConf.load("configs/config.yaml")
RAW_CSV = Path(cfg.paths.raw_data)
REPORTS_DIR = Path(cfg.paths.reports) / "validation"
engine = get_engine()

## 1. Load raw CSV

In [2]:
df = load_raw_csv(RAW_CSV)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"NULL totalcharges: {df['totalcharges'].isna().sum()} rows (zero-tenure customers — expected)")
df.head()

Shape: 7,043 rows × 21 columns
NULL totalcharges: 11 rows (zero-tenure customers — expected)


,customerid,gender,seniorcitizen,has_partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract_type,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [3]:
# The zero-tenure customers whose first bill has not yet been issued
df[df["totalcharges"].isna()][["customerid", "tenure", "monthlycharges", "totalcharges"]]

,customerid,tenure,monthlycharges,totalcharges
488,4472-LVYGI,0,52.55,NaN
753,3115-CZMZD,0,20.25,NaN
936,5709-LVOEQ,0,80.85,NaN
1082,4367-NUYAO,0,25.75,NaN
1340,1371-DWPAZ,0,56.05,NaN
3331,7644-OMVMY,0,19.85,NaN
3826,3213-VVOLG,0,25.35,NaN
4380,2520-SGTTA,0,20.00,NaN
5218,2923-ARZLG,0,19.70,NaN
6670,4075-WKNIU,0,73.35,NaN


## 2. Run the five data-quality gates

`validate_raw` is called here with `strict=False` so all gate results are visible
even if an ERROR gate fails. When `ingest()` runs in Section 3, it calls the same
function with `strict=True` — any ERROR gate blocks the load entirely.

In [4]:
result = validate_raw(df, strict=False, reports_dir=REPORTS_DIR)

### Gate results

In [5]:
status = "PASS — pipeline can proceed" if result.can_proceed else "FAIL — blocking errors found"
print(f"Overall: {status}")
print(f"Errors  : {len(result.errors)}")
print(f"Warnings: {len(result.warnings)}")

gate_summary = pd.DataFrame(
    [
        {
            "gate": c.name,
            "passed": c.passed,
            "severity": str(c.failure_severity),
            "affected_rows": c.affected_rows,
            "message": c.message,
        }
        for c in result.checks
    ]
)
display(gate_summary)

Overall: PASS — pipeline can proceed
Errors  : 0
Warnings: 0


,gate,passed,severity,affected_rows,message
0,schema,True,ERROR,0,Schema validation passed.
1,duplicate_ids,True,ERROR,0,No duplicate customerid values.
2,churn_labels,True,ERROR,0,"Churn labels are valid (binary, no missing val..."
3,totalcharges_unexpected_nulls,True,WARNING,0,No unexpected NULL totalcharges values.
4,row_count,True,WARNING,0,Row count 7043 meets the minimum threshold of ...
5,null_rate_churn,True,WARNING,0,'churn' null rate 0.0% is within threshold.
6,null_rate_contract_type,True,WARNING,0,'contract_type' null rate 0.0% is within thres...
7,null_rate_customerid,True,WARNING,0,'customerid' null rate 0.0% is within threshold.
8,null_rate_dependents,True,WARNING,0,'dependents' null rate 0.0% is within threshold.
9,null_rate_deviceprotection,True,WARNING,0,'deviceprotection' null rate 0.0% is within th...


### Failure details

The cells below mirror the `summary.csv` and `schema_failures.csv` written to
`reports/validation/` by `validate_raw`. They render as `None` when all gates pass.

In [6]:
# summary.csv equivalent — failing checks only
failing = [c for c in result.checks if not c.passed]

if not failing:
    print("summary.csv: no failing checks — file not written.")
else:
    summary_csv = pd.DataFrame(
        [
            {
                "check": c.name,
                "failure_severity": str(c.failure_severity),
                "message": c.message,
                "affected_rows": c.affected_rows,
            }
            for c in failing
        ]
    )
    print("summary.csv")
    display(summary_csv)

summary.csv: no failing checks — file not written.


In [7]:
# schema_failures.csv equivalent — row-level pandera failure cases
schema_check = next((c for c in result.checks if c.name == "schema"), None)

if schema_check is None or schema_check.passed:
    print("schema_failures.csv: schema check passed — file not written.")
else:
    print("schema_failures.csv")
    display(schema_check.detail)

schema_failures.csv: schema check passed — file not written.


## 3. Ingest to Postgres

`ingest()` is the production pipeline entry point. It:
1. Loads the CSV via `load_raw_csv()`
2. Validates with `validate_raw(strict=True)` — raises `ValidationError` and aborts on any ERROR gate
3. Creates `customers_raw` if it does not exist (preserving the `customerid` PRIMARY KEY)
4. Bulk-loads into a staging table, then merges via `INSERT … ON CONFLICT DO UPDATE`
5. Drops the staging table inside the same transaction
6. Asserts the DB-reported row count matches the CSV row count — raises `RuntimeError` if they differ, indicating a constraint violation or partial write

In [8]:
n = ingest(RAW_CSV, engine)
print(f"CSV rows : {len(df):,}")
print(f"DB rows  : {n:,}")
print(f"Status   : {'OK — counts match' if n == len(df) else 'MISMATCH — check logs'}")

{"csv_rows": 7043, "table": "customers_raw_staging", "event": "staging_loaded", "logger": "telco_churn.data.ingest", "level": "info", "timestamp": "2026-06-11T17:39:43.560829Z"}


{"db_rows": 7043, "csv_rows": 7043, "table": "customers_raw", "event": "merge_complete", "logger": "telco_churn.data.ingest", "level": "info", "timestamp": "2026-06-11T17:39:43.791685Z"}


CSV rows : 7,043
DB rows  : 7,043
Status   : OK — counts match


## 4. Confirm ingest

Three checks that go beyond what `ingest()` asserts internally:

- **Total table row count** — `ingest()` asserts the merge rowcount (rows the upsert touched) equals the CSV. `SELECT COUNT(*)` checks the total table state after the merge — a different guarantee: on re-ingest, a deletion bug elsewhere would not be caught by the merge count alone.
- **PRIMARY KEY constraint** — verifies the DDL constraint (and by extension the NOT NULL / CHECK constraints added in Phase 1 QA) actually landed on the table. `ingest()` does not verify DDL post-creation.
- **Sample rows** — confirms column names, types, and values are correct after the Postgres round-trip.

In [9]:
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM customers_raw")).scalar()
    has_pk = conn.execute(text("""
        SELECT COUNT(*) FROM information_schema.table_constraints
        WHERE table_name = 'customers_raw' AND constraint_type = 'PRIMARY KEY'
    """)).scalar()

print(f"Rows in customers_raw : {count:,}")
print(f"PRIMARY KEY present   : {bool(has_pk)}")

pd.read_sql("SELECT * FROM customers_raw LIMIT 5", engine)

Rows in customers_raw : 7,043
PRIMARY KEY present   : True


,customerid,gender,seniorcitizen,has_partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract_type,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,8665-UTDHZ,Male,0,Yes,Yes,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,30.20,30.20,1
1,5248-YGIJN,Male,0,Yes,No,72,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),90.25,6369.45,0
2,8773-HHUOZ,Female,0,No,Yes,17,Yes,No,DSL,No,...,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,64.70,1093.10,1
3,3841-NFECX,Female,1,Yes,No,71,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,No,No,Two year,Yes,Credit card (automatic),96.35,6766.95,0
4,4929-XIHVW,Male,1,Yes,No,2,Yes,No,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Credit card (automatic),95.50,181.65,0


---

## Summary

This notebook walks the full ingestion path: raw CSV → five data-quality gates →
`customers_raw` Postgres table. All gates pass on the IBM Telco dataset. The 11
NULL `totalcharges` rows are expected and flagged as a WARNING — they are handled
by the model training sklearn Pipeline, not here.

For a full narrative of the gate results, see
**[§1 Data Ingestion & Quality Checks](../ANALYSIS.md#1-data-ingestion--quality-checks)**
in `ANALYSIS.md`.